# CogVideoX-2B — Flying Animation

Generate a flying video using **CogVideoX-2B** (2.6B params, Tsinghua/ZhipuAI).

- **Model**: CogVideoX-2B (text-to-video) — high-quality open video generation
- **Output**: 13 frames at 720×480, ~1.6 seconds at 8fps
- **VRAM**: ~6GB (float16 + sequential CPU offload) — fits T4 free tier
- **Runtime**: T4 (free tier) works fine

> **Note**: CogVideoX-5B I2V (image-to-video) requires A100/Colab Pro — it exceeds T4's 12.7GB system RAM during loading. CogVideoX-2B uses text prompts instead. Frame count reduced from 49 to 13 to fit T4 RAM during inference.

In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors pillow imageio[ffmpeg] sentencepiece

import torch
print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} — {props.total_memory / 1024**3:.1f} GiB')
else:
    raise RuntimeError('No GPU!')
print('Dependencies installed.')

In [ ]:
# Cell 2: Load CogVideoX-2B text-to-video pipeline
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
import torch, gc

MODEL_ID = 'THUDM/CogVideoX-2b'

pipe = CogVideoXPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
)

pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

gc.collect()
torch.cuda.empty_cache()
print('CogVideoX-2B loaded with sequential CPU offload!')

In [ ]:
# Cell 3: Show character reference (text-to-video, no image conditioning)
import urllib.request
from PIL import Image

IMG_URL = 'https://raw.githubusercontent.com/mangeshgwagle/attestor/main/training/character_ref.webp'
IMG_PATH = '/content/character_ref.webp'
urllib.request.urlretrieve(IMG_URL, IMG_PATH)
ref_image = Image.open(IMG_PATH).convert('RGB')
print('Reference character (for prompt inspiration):')
display(ref_image)

In [ ]:
# Cell 4: Generate flying animation (memory-optimized for T4)
import torch, gc

PROMPT = (
    'A heroic angelic warrior character with golden armor and large white feathered wings '
    'takes flight, soaring upward through dramatic clouds. '
    'The wings spread wide as the character rises into a glowing celestial sky with golden light. '
    'Wind flows through their flowing cape. Epic fantasy cinematic, high quality, smooth motion.'
)

gc.collect()
torch.cuda.empty_cache()

generator = torch.manual_seed(42)

video_frames = pipe(
    prompt=PROMPT,
    num_frames=13,
    guidance_scale=6.0,
    num_inference_steps=30,
    generator=generator,
).frames[0]

print(f'Generated {len(video_frames)} frames!')

In [ ]:
# Cell 5: Export to MP4 and display inline
from diffusers.utils import export_to_video
from IPython.display import HTML
from base64 import b64encode
import os

MP4_PATH = '/content/cogvideo_flying.mp4'
export_to_video(video_frames, MP4_PATH, fps=8)

mp4_size = os.path.getsize(MP4_PATH) / 1024**2
print(f'Video: {mp4_size:.1f} MB, {len(video_frames)} frames at 8fps = {len(video_frames)/8:.1f}s')

with open(MP4_PATH, 'rb') as f:
    mp4 = b64encode(f.read()).decode()
html_str = (
    '<video width="720" controls autoplay loop>'
    + '<source src="data:video/mp4;base64,' + mp4 + '" type="video/mp4">'
    + '</video>'
)
display(HTML(html_str))
print('Video is playing above!')

In [ ]:
# Cell 6: Generate variant — battle dive
import torch, gc

PROMPT_V2 = (
    'A golden-armored angelic warrior diving through storm clouds at high speed, '
    'wings folded tight for a dive, glowing with celestial energy, '
    'lightning flashing around them, dramatic action scene, '
    'fantasy game cinematic, motion blur, high quality, smooth motion.'
)

gc.collect()
torch.cuda.empty_cache()

generator = torch.manual_seed(123)
video_frames_v2 = pipe(
    prompt=PROMPT_V2,
    num_frames=13,
    guidance_scale=6.0,
    num_inference_steps=30,
    generator=generator,
).frames[0]

V2_PATH = '/content/cogvideo_flying_v2.mp4'
export_to_video(video_frames_v2, V2_PATH, fps=8)
print(f'Variant 2: {os.path.getsize(V2_PATH)/1024**2:.1f} MB')

In [ ]:
# Cell 7: Download all videos
from google.colab import files
import glob, os

for f in sorted(glob.glob('/content/cogvideo_*.mp4')):
    size = os.path.getsize(f) / 1024**2
    print(f'Downloading {os.path.basename(f)} ({size:.1f} MB)...')
    files.download(f)

print('Done! Check the Files sidebar (folder icon) if downloads did not trigger.')